# 00 — Exploratory Data Analysis & Noise Discovery

**Phase 0, Steps 0.3 & 0.4** — Systematic EDA with programmatic profiling.

Sections:
1. Setup & data loading (sample-based for speed)
2. Schema inspection — column names, dtypes, missing rates
3. Country distribution
4. Character-set profiling — **fully dynamic** Unicode script discovery
5. Token frequency analysis
6. Regex noise scans with counts
7. Missing field rates by country
8. Cross-script match analysis (ground truth pairs)
9. Noise catalog — prioritised summary

In [27]:
from __future__ import annotations
import os, re, warnings, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import regex  # used for tokenization in Section 5

warnings.filterwarnings("ignore")

# --- Paths -------------------------------------------------------------------
REPO_ROOT = Path(os.getcwd())
for p in [Path(os.getcwd()), *Path(os.getcwd()).parents]:
    if (p / "dataset").exists():
        REPO_ROOT = p
        break

TRAIN_DIR = REPO_ROOT / "dataset" / "train"
TEST_DIR  = REPO_ROOT / "dataset" / "test"

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"TRAIN_DIR  : {TRAIN_DIR}")
print(f"TEST_DIR   : {TEST_DIR}")
assert TRAIN_DIR.exists(), f"Train dir not found: {TRAIN_DIR}"
assert TEST_DIR.exists(),  f"Test dir not found:  {TEST_DIR}"

REPO_ROOT  : c:\Users\arula\Documents\Work\ML_challenge
TRAIN_DIR  : c:\Users\arula\Documents\Work\ML_challenge\dataset\train
TEST_DIR   : c:\Users\arula\Documents\Work\ML_challenge\dataset\test


In [19]:
# ---------------------------------------------------------------------------
# Sampling configuration
# Full files are 200-500 MB each. We sample for interactive EDA.
# ---------------------------------------------------------------------------
SAMPLE_N    = 200_000   # rows per source for interactive analysis
RANDOM_SEED = 42

def load_tsv_sample(path, n=SAMPLE_N, seed=RANDOM_SEED):
    """Load a random sample of rows from a TSV file efficiently."""
    total = sum(1 for _ in open(path, "r", encoding="utf-8")) - 1  # minus header
    print(f"  {path.name}: {total:,} rows  ->  sampling {min(n, total):,}")

    if total <= n:
        return pd.read_csv(path, sep="\t", dtype=str, low_memory=False)

    rng = np.random.default_rng(seed)
    skip_idx = rng.choice(total, total - n, replace=False)
    skip_set = set(skip_idx + 1)  # +1 because header is row 0
    return pd.read_csv(
        path, sep="\t", dtype=str, low_memory=False,
        skiprows=lambda i: i in skip_set,
    )

print("Loading samples ...")
s1_train = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", dtype=str, low_memory=False)
s2_train = pd.read_csv(TRAIN_DIR / "train_source2.tsv", sep="\t", dtype=str, low_memory=False)
s3_train = pd.read_csv(TRAIN_DIR / "train_source3.tsv", sep="\t", dtype=str, low_memory=False)
print("Done.")

Loading samples ...
Done.


## 1. Schema inspection

In [21]:
for name, df in [("S1 (train)", s1_train), ("S2 (train)", s2_train), ("S3 (train)", s3_train)]:
    print(f"\n--- {name} ({len(df):,} sampled rows) ---")
    print(f"Columns: {list(df.columns)}")
    print(df.dtypes)
    print("\nMissing rates:")
    print((df.isna() | (df == "")).mean().to_string())


--- S1 (train) (2,206,821 sampled rows) ---
Columns: ['entity_id', 'business_name', 'business_address', 'country']
entity_id           str
business_name       str
business_address    str
country             str
dtype: object

Missing rates:
entity_id           0.0
business_name       0.0
business_address    0.0
country             0.0

--- S2 (train) (5,034,616 sampled rows) ---
Columns: ['entity_id', 'business_name', 'business_address', 'country']
entity_id           str
business_name       str
business_address    str
country             str
dtype: object

Missing rates:
entity_id           0.000000e+00
business_name       3.972498e-07
business_address    3.356105e-02
country             0.000000e+00

--- S3 (train) (5,285,603 sampled rows) ---
Columns: ['entity_id', 'business_name', 'business_address', 'country']
entity_id           str
business_name       str
business_address    str
country             str
dtype: object

Missing rates:
entity_id           0.000000
business_name    

## 2. Country distribution

In [22]:
for name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    col = "country" if "country" in df.columns else None
    if col:
        print(f"\n{name} country distribution:")
        print(df[col].value_counts(dropna=False).to_string())


S1 country distribution:
country
US       1323633
India     883188

S2 country distribution:
country
US       3016817
India    2017799

S3 country distribution:
country
US       3170056
India    2115547


## 3. Character-set profiling — fully dynamic Unicode script discovery

### 3a. Character-level inventory

We make **no assumptions** about which scripts exist. We scan every unique
non-ASCII character and use `unicodedata.name()` to extract the script/block
family automatically. No predefined list.

In [23]:
NAME_COL = "business_name"
ADDR_COL = "business_address"


def build_script_inventory(df, col):
    """
    Dynamically discover every Unicode script/block present in `col`.

    1. Collect all unique non-ASCII characters across the column.
    2. unicodedata.name(ch) -> e.g. "DEVANAGARI LETTER RA"
    3. First word = script/block family.  No hardcoded list.

    Returns
    -------
    script_counts : Counter  {script_name: distinct_char_count}
    sample_chars  : dict     {script_name: [up to 8 sample chars]}
    """
    all_text  = "".join(df[col].dropna().astype(str))
    non_ascii = {ch for ch in all_text if ord(ch) >= 128}

    script_counts = Counter()
    sample_chars  = {}
    unknown_chars = []

    for ch in non_ascii:
        uname  = unicodedata.name(ch, "UNKNOWN")
        script = uname.split()[0] if uname != "UNKNOWN" else "UNKNOWN"

        script_counts[script] += 1
        if len(sample_chars.get(script, [])) < 8:
            sample_chars.setdefault(script, []).append(ch)

        if uname == "UNKNOWN":
            unknown_chars.append(f"U+{ord(ch):04X}")

    if unknown_chars:
        print(f"  WARNING: {len(unknown_chars)} chars had no Unicode name: {unknown_chars[:10]}")

    return script_counts, sample_chars


# --- Run per source ----------------------------------------------------------
SCRIPT_INVENTORIES = {}

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        print(f"{src_name}: no '{NAME_COL}' column"); continue

    counts, samples = build_script_inventory(df, NAME_COL)
    SCRIPT_INVENTORIES[src_name] = (counts, samples)

    print(f"\n{src_name} -- Unicode script/block inventory ({len(counts)} families found):")
    rows = []
    for script, n_chars in counts.most_common():
        chars_display = " ".join(samples.get(script, []))
        rows.append({"script/block": script, "distinct_chars": n_chars, "samples": chars_display})
    inv_df = pd.DataFrame(rows)
    print(inv_df.to_string(index=False))


S1 -- Unicode script/block inventory (0 families found):
Empty DataFrame
Columns: []
Index: []

S2 -- Unicode script/block inventory (11 families found):
script/block  distinct_chars         samples
   MALAYALAM              54 ം ോ സ ൂ ഐ ഹ ഫ ഴ
  DEVANAGARI              46 ष ई र ॐ प न ट च
     KANNADA              46 ಐ ಹ ದ ್ ಾ ಕ ೋ ಿ
    GUJARATI              45 ઈ ઇ ચ થ ા ુ પ ડ
      TELUGU              45 గ ద ప ా ి ఐ డ ృ
    GURMUKHI              44 ਤ ਨ ਼ ੱ ਰ ੈ ੰ ਚ
       ORIYA              44 ୍ ଜ ପ ଡ ଉ ଂ ଇ ଓ
     BENGALI              42 ৈ এ ল থ ঞ ূ ু শ
       TAMIL              41 ஹ இ ஈ ந ர ஷ ை ங
       LATIN              10 í Á á É ú ó Ó Í
        ZERO               1               ‌

S3 -- Unicode script/block inventory (11 families found):
script/block  distinct_chars         samples
   MALAYALAM              54 ം ോ സ ൂ ഐ ഹ ഫ ഴ
  DEVANAGARI              46 ष ई र ॐ प न ट ँ
     KANNADA              46 ಐ ಹ ದ ್ ಾ ಕ ೋ ಿ
    GUJARATI              45 ઈ ઇ ચ થ ા ુ પ ડ
      TELUGU         

### 3b. Record-level script classifier (fully dynamic)

Uses `unicodedata.category()` (Unicode standard) to identify **letters/marks**
(script-bearing) vs punctuation/symbols/digits (structural).
Then `unicodedata.name()` extracts the script.

Only assumption: `LATIN` is the baseline. Everything else is discovered.
**No hardcoded script list whatsoever.**

In [24]:
# Unicode General Categories that carry script identity:
#   L* = Letters,  M* = Marks (diacritics/combining, tied to a script)
# Everything else (P*, S*, N*, Z*, C*) is structural.
SCRIPT_BEARING_CATEGORIES = {"L", "M"}


def get_char_script(ch):
    """Return the script family for a letter/mark character, or None for structural."""
    if ord(ch) < 128:
        return None  # ASCII — skip
    cat = unicodedata.category(ch)
    if cat[0] not in SCRIPT_BEARING_CATEGORIES:
        return None  # punctuation, symbol, digit, separator, control
    uname = unicodedata.name(ch, "")
    if not uname:
        return None
    return uname.split()[0]


def detect_scripts_dynamic(text):
    """
    Return the set of all script families for letter/mark characters in text.
    Purely data-driven. No predefined script list.
    """
    if not isinstance(text, str) or not text.strip():
        return set()
    scripts = set()
    for ch in text:
        s = get_char_script(ch)
        if s is not None:
            scripts.add(s)
    return scripts


def is_non_latin_name(text):
    """True if the name contains letter/mark characters from any non-LATIN script."""
    scripts = detect_scripts_dynamic(text)
    scripts.discard("LATIN")
    return len(scripts) > 0


# --- Record-level stats per source -------------------------------------------
for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        continue

    non_latin_mask = df[NAME_COL].apply(is_non_latin_name)
    pct = non_latin_mask.mean() * 100
    print(f"\n{src_name}: {non_latin_mask.sum():,} / {len(df):,} ({pct:.2f}%) records contain non-Latin script chars")

    script_record_counter = Counter()
    mixed_script_count = 0
    for text in df.loc[non_latin_mask, NAME_COL]:
        scripts = detect_scripts_dynamic(text)
        scripts.discard("LATIN")
        for s in scripts:
            script_record_counter[s] += 1
        if len(scripts) > 1:
            mixed_script_count += 1

    print("  Discovered non-Latin scripts (records containing each):")
    for script, cnt in script_record_counter.most_common():
        print(f"    {script:<20}: {cnt:>7,}")
    print(f"  Mixed non-Latin script records (2+ scripts in one name): {mixed_script_count:,}")


S1: 0 / 2,206,821 (0.00%) records contain non-Latin script chars
  Discovered non-Latin scripts (records containing each):
  Mixed non-Latin script records (2+ scripts in one name): 0

S2: 474,345 / 5,034,616 (9.42%) records contain non-Latin script chars
  Discovered non-Latin scripts (records containing each):
    DEVANAGARI          : 269,424
    TELUGU              :  39,323
    KANNADA             :  37,211
    TAMIL               :  33,781
    GUJARATI            :  30,929
    BENGALI             :  30,723
    MALAYALAM           :  18,773
    ORIYA               :   7,493
    GURMUKHI            :   6,688
  Mixed non-Latin script records (2+ scripts in one name): 0

S3: 278,524 / 5,285,603 (5.27%) records contain non-Latin script chars
  Discovered non-Latin scripts (records containing each):
    DEVANAGARI          : 158,003
    TELUGU              :  23,033
    KANNADA             :  21,995
    TAMIL               :  19,790
    BENGALI             :  18,144
    GUJARATI      

## 4. Token frequency analysis — top tokens & legal suffix inventory

In [25]:
def tokenize_latin(text):
    if not isinstance(text, str):
        return []
    return regex.findall(r"[a-zA-Z0-9]+", text.lower())

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        continue
    counter = Counter()
    for text in df[NAME_COL].dropna():
        counter.update(tokenize_latin(text))
    print(f"\n{src_name} -- Top 30 business_name tokens:")
    print(counter.most_common(30))
    print(f"  Total unique tokens: {len(counter):,}")


S1 -- Top 30 business_name tokens:
[('limited', 522340), ('private', 432503), ('llc', 355736), ('inc', 238309), ('ltd', 148599), ('pvt', 121462), ('india', 61241), ('and', 57296), ('s', 55791), ('c', 54031), ('care', 53136), ('l', 49410), ('of', 44497), ('associates', 42985), ('group', 40520), ('llp', 39726), ('center', 35433), ('partners', 35084), ('corp', 34792), ('services', 31242), ('p', 29137), ('health', 28264), ('d', 28059), ('co', 27692), ('clinic', 27586), ('pc', 27166), ('solutions', 26415), ('global', 25611), ('trading', 22806), ('international', 20843)]
  Total unique tokens: 108,857

S2 -- Top 30 business_name tokens:
[('llc', 532379), ('limited', 532213), ('private', 532080), ('inc', 403242), ('ltd', 401133), ('com', 202143), ('l', 200498), ('center', 196536), ('pvt', 190499), ('s', 155701), ('partners', 155487), ('c', 148164), ('services', 147345), ('corp', 142833), ('group', 141459), ('co', 140923), ('and', 118901), ('india', 114479), ('holdings', 97736), ('of', 89957)

In [ ]:
suffix_counts = Counter()
suffix_examples = defaultdict(list)

for src_name, df in [
    ("S1", s1_train),
    ("S2", s2_train),
    ("S3", s3_train),
]:
    if NAME_COL not in df.columns:
        continue

    for value in df[NAME_COL].dropna():
        if not isinstance(value, str):
            continue

        m = LEGAL_RE.search(value.strip())

        if m:
            suffix = m.group(0).lower()
            suffix_counts[suffix] += 1

            # Keep only a few examples per suffix
            if len(suffix_examples[suffix]) < 5:
                suffix_examples[suffix].append(value)

print(f"Unique suffixes found: {len(suffix_counts):,}")

for suffix, count in suffix_counts.most_common():
    print(f"{suffix:30} {count:>10,}")

Unique suffixes found: 50
limited               1,407,293
llc                   1,153,852
ltd                     713,826
inc                     644,229
corp                    242,921
private                 228,748
co                      219,667
services                203,559
partners                199,729
group                   187,473
llp                     141,145
holdings                124,934
corporation             110,897
lp                       93,710
company                  55,388
pllc                     55,177
pvt                      51,787
clinic                   48,609
enterprises              31,297
trust                    28,309
ventures                 23,723
industries               22,847
society                  19,546
foundation               18,954
solutions                15,996
school                   14,623
institute                12,851
association              12,744
technologies             12,214
academy                  11,965
international 

In [34]:
# Legal suffix inventory
LEGAL_RE = regex.compile(
    r"\b(llc|ltd|limited|inc|corp|corporation|pvt|private|llp|lp|co|company|"
    r"sarl|sci|sas|plc|pllc|gmbh|bv|ag|sa|sl|srl|pty|ngo|npo|trust|"
    r"foundation|association|society|institute|university|college|school|"
    r"hospital|clinic|academy|services|solutions|group|holdings|enterprises|"
    r"international|industries|consulting|technologies|tech|labs|media|"
    r"ventures|partners|capital)$",
    flags=regex.IGNORECASE,
)

def get_legal_suffix(text):
    if not isinstance(text, str):
        return None
    m = LEGAL_RE.search(text.strip())
    return m.group(0).lower() if m else None

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        continue
    suffixes = df[NAME_COL].apply(get_legal_suffix)
    print(f"\n{src_name} -- Legal suffix distribution:")
    print(suffixes.value_counts(dropna=False).head(20).to_string())


S1 -- Legal suffix distribution:
business_name
NaN            704679
limited        521915
llc            355736
inc            190223
ltd            122056
llp             39725
corp            33761
group           32678
co              25178
pllc            20531
partners        19351
clinic          18107
lp              16477
corporation     13540
company         12316
trust            9924
holdings         8309
society          7024
school           5913
foundation       5313

S2 -- Legal suffix distribution:
business_name
NaN            2821055
limited         389262
llc             381300
ltd             286395
inc             218146
private         111811
corp            105338
co               97798
services         92007
partners         85395
group            75948
holdings         57885
corporation      50886
llp              46289
lp               39789
pvt              25242
company          21522
pllc             15911
clinic           14144
enterprises      12541

S3 

## 5. Regex noise scans with counts

In [36]:
NOISE_PATTERNS = {
    "hashtag_prefix":     re.compile(r"^[#<\s]+"),
    "appended_digits":    re.compile(r"[-\s]\d{6,}\s*$|#\d{5,}\s*$"),
    "url_in_name":        re.compile(r"www\.|\.(com|net|org|in|co\.in)|https?://|@\w+"),
    "duplicate_adj_word": re.compile(r"\b(\w{3,})\s+\1\b", re.IGNORECASE),
    "extra_whitespace":   re.compile(r"\s{2,}"),
    "null_literal":       re.compile(r"\bnull\b", re.IGNORECASE),
    "angle_brackets":     re.compile(r"<<|>>"),
    "all_caps_word":      re.compile(r"\b[A-Z]{4,}\b"),
    "dba_alias":          re.compile(r"\bdba\b", re.IGNORECASE),
    "parenthetical_sfx":  re.compile(r"\((?:LLC|LLP|Inc|Corp|Co|Ltd|Limited|Pvt|Private)\)", re.IGNORECASE),
    "accent_anomaly":     re.compile(r"[\u00c0-\u00ff]"),
    "pipe_separator":     re.compile(r"\|"),
    "leetspeak_digit":    re.compile(r"\d[a-zA-Z]|[a-zA-Z]\d"),   # e.g. "5uperior"
}

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        continue
    col = df[NAME_COL].fillna("")
    print(f"\n{src_name} -- Noise pattern scan on business_name (n={len(df):,}):")
    for pat_name, pat in NOISE_PATTERNS.items():
        mask = col.str.contains(pat, na=False)
        n = mask.sum()
        pct = n / len(df) * 100
        if n > 0:
            example = col[mask].iloc[0]
            print(f"  {pat_name:<22}: {n:>7,} ({pct:5.2f}%)  e.g. {repr(example[:80])}")


S1 -- Noise pattern scan on business_name (n=2,206,821):
  hashtag_prefix        :   1,505 ( 0.07%)  e.g. '#4 Presidio'
  appended_digits       :       2 ( 0.00%)  e.g. 'Swastik Finvest-Chennai 600008'
  duplicate_adj_word    :   2,741 ( 0.12%)  e.g. 'Surgical Clinic Clinic'
  null_literal          :      20 ( 0.00%)  e.g. 'Leblanc, Null & Mohl Arbor LLC'
  all_caps_word         :  20,531 ( 0.93%)  e.g. 'Liberty Family Office PLLC'
  dba_alias             :       6 ( 0.00%)  e.g. 'Dba Construction Pvt Ltd'
  parenthetical_sfx     :       5 ( 0.00%)  e.g. 'Hind (private) Private Limited'
  leetspeak_digit       :     605 ( 0.03%)  e.g. '24HR Towing'

S2 -- Noise pattern scan on business_name (n=5,034,616):
  hashtag_prefix        :  17,779 ( 0.35%)  e.g. '#hanumaneducational'
  appended_digits       :  29,356 ( 0.58%)  e.g. 'Diamond Star Works L.L.C. - 7149969359'
  url_in_name           : 176,144 ( 3.50%)  e.g. 'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com'

In [37]:
# Address-specific noise
ADDR_PATTERNS = {
    "null_literal":    re.compile(r"\bnull\b", re.IGNORECASE),
    "po_box":          re.compile(r"\bP\.?O\.?\s*Box\b", re.IGNORECASE),
    "all_caps":        re.compile(r"^[A-Z0-9\s,\.#-]+$"),
    "reordered_addr":  re.compile(r"^[A-Z]{2},"),  # state before street
    "empty":           re.compile(r"^\s*$"),
    "non_latin_chars": re.compile(r"[^\x00-\x7f]"),  # any non-ASCII in address
}

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if ADDR_COL not in df.columns:
        continue
    col = df[ADDR_COL].fillna("")
    print(f"\n{src_name} -- Address noise scan (n={len(df):,}):")
    for pat_name, pat in ADDR_PATTERNS.items():
        if pat_name in ("all_caps", "reordered_addr", "empty"):
            mask = col.str.match(pat)
        else:
            mask = col.str.contains(pat, na=False)
        n = mask.sum()
        pct = n / len(df) * 100
        print(f"  {pat_name:<20}: {n:>7,} ({pct:5.2f}%)")


S1 -- Address noise scan (n=2,206,821):
  null_literal        :      43 ( 0.00%)
  po_box              :      15 ( 0.00%)
  all_caps            :       4 ( 0.00%)
  reordered_addr      :  84,634 ( 3.84%)
  empty               :       0 ( 0.00%)
  non_latin_chars     :     554 ( 0.03%)

S2 -- Address noise scan (n=5,034,616):
  null_literal        : 131,794 ( 2.62%)
  po_box              :  50,273 ( 1.00%)
  all_caps            : 2,598,431 (51.61%)
  reordered_addr      : 191,054 ( 3.79%)
  empty               : 168,967 ( 3.36%)
  non_latin_chars     : 478,453 ( 9.50%)

S3 -- Address noise scan (n=5,285,603):
  null_literal        : 130,844 ( 2.48%)
  po_box              :  49,818 ( 0.94%)
  all_caps            :     408 ( 0.01%)
  reordered_addr      :  76,508 ( 1.45%)
  empty               : 175,916 ( 3.33%)
  non_latin_chars     : 476,588 ( 9.02%)


## 6. Missing field rates by country

In [38]:
for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if "country" not in df.columns or ADDR_COL not in df.columns:
        continue
    missing = (
        df.assign(_empty=(df[ADDR_COL].isnull() | (df[ADDR_COL].str.strip() == "")))
          .groupby("country")["_empty"]
          .agg(["sum", "count", "mean"])
          .rename(columns={"sum": "empty", "count": "total", "mean": "pct_empty"})
    )
    print(f"\n{src_name} -- address missing rate by country:")
    print(missing.to_string())


S1 -- address missing rate by country:
         empty    total  pct_empty
country                           
India        0   883188        0.0
US           0  1323633        0.0

S2 -- address missing rate by country:
          empty    total  pct_empty
country                            
India     57846  2017799   0.028668
US       111121  3016817   0.036834

S3 -- address missing rate by country:
          empty    total  pct_empty
country                            
India     64948  2115547   0.030700
US       110968  3170056   0.035005


## 7. Cross-script match analysis (ground truth pairs)

Quantifies how many ground-truth (S1, S2/S3) pairs involve a script mismatch.
Uses `detect_scripts_dynamic()` from Section 3b — fully data-driven.

In [39]:
gt_path = TRAIN_DIR / "train_ground_truth.tsv"
if not gt_path.exists():
    print("Ground truth file not found -- skipping cross-script analysis")
else:
    print("Loading ground truth sample ...")
    gt_sample = load_tsv_sample(gt_path, n=50_000)
    print(f"  Loaded {len(gt_sample):,} GT rows")
    print(f"  Columns: {list(gt_sample.columns)}")

    # Singletons
    has_match = gt_sample["matched_entity_ids"].notna() & (gt_sample["matched_entity_ids"].str.strip() != "")
    print(f"\n  Singletons (no match): {(~has_match).sum():,} / {len(gt_sample):,} ({(~has_match).mean()*100:.2f}%)")

    # Match count distribution
    match_counts = gt_sample.loc[has_match, "matched_entity_ids"].str.split(",").str.len()
    print(f"\n  Match count distribution (non-singletons):")
    print(match_counts.value_counts().sort_index().head(15).to_string())
    print(f"  Avg matches per non-singleton: {match_counts.mean():.2f}")
    print(f"  Max matches: {match_counts.max()}")

Loading ground truth sample ...
  train_ground_truth.tsv: 2,206,821 rows  ->  sampling 50,000
  Loaded 50,000 GT rows
  Columns: ['source1_entity_id', 'matched_entity_ids']

  Singletons (no match): 2,847 / 50,000 (5.69%)

  Match count distribution (non-singletons):
matched_entity_ids
1      2698
2      8600
3     11919
4     10936
5      7210
6      3759
7      1486
8       435
9        96
10       11
11        3
  Avg matches per non-singleton: 3.67
  Max matches: 11


In [40]:
# Cross-script: uses detect_scripts_dynamic() -- no hardcoded script list.
if gt_path.exists() and "gt_sample" in dir():
    # Build name lookups from samples
    id_col = "entity_id"

    s1_name_map = s1_train.set_index(id_col)[NAME_COL].to_dict() if NAME_COL in s1_train.columns else {}
    s2_name_map = s2_train.set_index(id_col)[NAME_COL].to_dict() if NAME_COL in s2_train.columns else {}
    s3_name_map = s3_train.set_index(id_col)[NAME_COL].to_dict() if NAME_COL in s3_train.columns else {}

    cross_script        = 0
    total_pairs         = 0
    script_pair_counter = Counter()  # (s1_scripts, s23_scripts) -> count

    for _, row in gt_sample.iterrows():
        s1_id      = row.get("source1_entity_id", "")
        s1_name    = s1_name_map.get(s1_id, "")
        s1_scripts = frozenset(detect_scripts_dynamic(s1_name)) or frozenset(["LATIN"])

        raw = row.get("matched_entity_ids", "")
        if not isinstance(raw, str) or not raw.strip():
            continue

        for mid in raw.split(","):
            mid = mid.strip()
            if   mid.startswith("S2-"): matched_name = s2_name_map.get(mid, "")
            elif mid.startswith("S3-"): matched_name = s3_name_map.get(mid, "")
            else: continue

            total_pairs  += 1
            s23_scripts   = frozenset(detect_scripts_dynamic(matched_name)) or frozenset(["LATIN"])

            if s1_scripts != s23_scripts:
                cross_script += 1
                pair_key = (tuple(sorted(s1_scripts)), tuple(sorted(s23_scripts)))
                script_pair_counter[pair_key] += 1

    if total_pairs > 0:
        print(f"Cross-script pairs: {cross_script:,} / {total_pairs:,} ({cross_script/total_pairs*100:.2f}%)")
        print("  (note: limited by sampled lookups -- not all matched IDs may be in sample)")
        print("\n  Top cross-script pair types:")
        for (s1_s, s23_s), cnt in script_pair_counter.most_common(10):
            print(f"    S1:{list(s1_s)} <-> S2/S3:{list(s23_s)}  -- {cnt:,} pairs")
    else:
        print("No matching pairs found in sample overlap -- increase SAMPLE_N or run on full data")

Cross-script pairs: 12,557 / 172,892 (7.26%)
  (note: limited by sampled lookups -- not all matched IDs may be in sample)

  Top cross-script pair types:
    S1:['LATIN'] <-> S2/S3:['DEVANAGARI']  -- 7,120 pairs
    S1:['LATIN'] <-> S2/S3:['TELUGU']  -- 1,116 pairs
    S1:['LATIN'] <-> S2/S3:['KANNADA']  -- 957 pairs
    S1:['LATIN'] <-> S2/S3:['BENGALI']  -- 897 pairs
    S1:['LATIN'] <-> S2/S3:['TAMIL']  -- 890 pairs
    S1:['LATIN'] <-> S2/S3:['GUJARATI']  -- 812 pairs
    S1:['LATIN'] <-> S2/S3:['MALAYALAM']  -- 470 pairs
    S1:['LATIN'] <-> S2/S3:['ORIYA']  -- 162 pairs
    S1:['LATIN'] <-> S2/S3:['GURMUKHI']  -- 133 pairs


## 7b. Catch-All Unknown Pattern Discovery

Dynamically discover high-frequency anomalous tokens not covered by predefined patterns.
This catches unexpected noise patterns specific to this dataset.

In [41]:
def find_anomalous_tokens(df, col):
    """Find suspiciously high-frequency non-word tokens and mixed alphanumeric patterns."""
    tokens = []
    for text in df[col].dropna():
        if not isinstance(text, str):
            continue
        # Extract non-standard tokens:
        # - Pure symbol sequences (e.g., "###", "|||")
        # - Mixed digit-letter patterns (e.g., "5uperior", "H3llo")
        # - Repeated punctuation
        tokens.extend(regex.findall(r"[^\w\s]{2,}|\d+[a-z]+|[a-z]+\d+|\w*[^\w\s]+\w+", text.lower()))
    
    counter = Counter(tokens)
    # Flag tokens appearing in >0.5% of records
    threshold = len(df) * 0.005
    return {tok: cnt for tok, cnt in counter.items() if cnt > threshold}

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        continue
    print(f"{src_name} -- High-frequency anomalous tokens (candidates for new noise rules):")
    anomalies = find_anomalous_tokens(df, NAME_COL)
    if anomalies:
        for tok, cnt in sorted(anomalies.items(), key=lambda x: -x[1])[:20]:
            pct = cnt / len(df) * 100
            # Find an example
            example_mask = df[NAME_COL].str.contains(regex.escape(tok), na=False, case=False)
            example = df.loc[example_mask, NAME_COL].iloc[0] if example_mask.any() else ""
            print(f"  {repr(tok):<25}: {cnt:>7,} ({pct:5.2f}%)  e.g. {repr(example[:60])}")
    else:
        print("  No high-frequency anomalous tokens found (threshold: >0.5% of records)")

S1 -- High-frequency anomalous tokens (candidates for new noise rules):
  '(india'                 :  44,262 ( 2.01%)  e.g. 'Osprey (India) Development LLP'
  'p.c'                    :  23,272 ( 1.05%)  e.g. 'WG Harbor Worldwide P.C.'
  '.c'                     :  21,855 ( 0.99%)  e.g. 'Aggie E. Nagle, L.C.S.W.'
  'l.l'                    :  21,843 ( 0.99%)  e.g. 'Petronilla Sult Royalties L.L.C.'
  '.,'                     :  19,086 ( 0.86%)  e.g. 'White, Helge W., DDS'
S2 -- High-frequency anomalous tokens (candidates for new noise rules):
  '(india'                 :  80,713 ( 1.60%)  e.g. 'GM (INDIA) FABRICS LTD'
  '.c'                     :  73,638 ( 1.46%)  e.g. 'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.sh'
  'l.l'                    :  72,731 ( 1.44%)  e.g. 'Piedmont Complete Chile  L.L.C.'
  '.,'                     :  34,215 ( 0.68%)  e.g. 'Kalie Cassidy, O.D.,'
  'p.c'                    :  26,678 ( 0.53%)  e.g. 'SHAFFER, LOVE + HALL LIGAND P.C. #98587'
  'm

## 7c. Verification Pass � Full Data Frequency Check

**Purpose**: Verify that pattern frequencies observed in the 200k sample are
representative of the full dataset. Critical patterns that will drive normalization
rules are re-counted on the complete data.

**Sampling strategy**: Initial EDA used stratified random samples of 200k records
per source (~4-9% of training data). This verification pass confirms that
high-frequency patterns (>1%) have stable frequency estimates before building
normalization rules.

In [42]:
import csv

def verify_pattern_on_full_data(path, patterns, col_name="business_name"):
    """Stream full TSV file and count pattern matches without loading into memory.
    Parameters
    ----------
    path : Path
        Path to TSV file
    patterns : dict
        {pattern_name: compiled regex}
    col_name : str
        Column name to scan
    
    Returns
    -------
    dict : {pattern_name: (count, percentage)}
    """
    results = {name: 0 for name in patterns.keys()}
    total_rows = 0
    
    print(f"  Streaming {path.name} ...")
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            total_rows += 1
            text = row.get(col_name, "")
            for pat_name, pat in patterns.items():
                if pat.search(text):
                    results[pat_name] += 1
            
            # Progress indicator every 500k rows
            if total_rows % 500_000 == 0:
                print(f"    ... processed {total_rows:,} rows")
    
    return {name: (count, count/total_rows*100 if total_rows > 0 else 0.0) 
            for name, count in results.items()}, total_rows

# Select patterns with >1% frequency in sample to verify on full data
VERIFY_PATTERNS = {
    name: pat for name, pat in NOISE_PATTERNS.items()
    # Add patterns here that showed >1% in sample - will be determined after first EDA run
}

# By default, verify the most impactful patterns
CRITICAL_PATTERNS = {
    "url_in_name":        NOISE_PATTERNS["url_in_name"],
    "duplicate_adj_word": NOISE_PATTERNS["duplicate_adj_word"],
    "all_caps_word":      NOISE_PATTERNS["all_caps_word"],
    "extra_whitespace":   NOISE_PATTERNS["extra_whitespace"],
    "dba_alias":          NOISE_PATTERNS["dba_alias"],
}

print("" + "="*80)
print("VERIFICATION PASS: Pattern frequencies on FULL training data")
print("="*80)
print("This confirms that sample-based frequencies are representative.")
print("Only run this after initial EDA to verify patterns you plan to normalize.")

# Verify on Source 2 (noisiest source)
s2_path = TRAIN_DIR / "train_source2.tsv"
if s2_path.exists():
    print(f"Verifying on {s2_path.name} (full file)...")
    full_results, total = verify_pattern_on_full_data(s2_path, CRITICAL_PATTERNS, NAME_COL)
    
    print(f"Full data pattern frequencies (n={total:,} rows):")
    print(f"{'Pattern':<22}  {'Count':>10}  {'Percentage':>10}  {'Sample vs Full'}")
    print("-" * 75)
    for pat_name, (count, pct) in sorted(full_results.items(), key=lambda x: -x[1][1]):
        # Compare to sample if available
        comparison = ""
        if NAME_COL in s2_train.columns:
            sample_mask = s2_train[NAME_COL].fillna("").str.contains(
                CRITICAL_PATTERNS[pat_name], na=False
            )
            sample_pct = sample_mask.mean() * 100
            diff = pct - sample_pct
            comparison = f"{diff:+.2f}% diff"
        
        print(f"{pat_name:<22}: {count:>10,}  {pct:>9.2f}%  {comparison}")
    
    print(f"? Verification complete. Update noise_catalog with full-data frequencies.")
else:
    print(f"? {s2_path} not found - skipping full-data verification.")
    print("  Run this cell after initial EDA to confirm pattern frequencies.")

VERIFICATION PASS: Pattern frequencies on FULL training data
This confirms that sample-based frequencies are representative.
Only run this after initial EDA to verify patterns you plan to normalize.
Verifying on train_source2.tsv (full file)...
  Streaming train_source2.tsv ...
    ... processed 500,000 rows
    ... processed 1,000,000 rows
    ... processed 1,500,000 rows
    ... processed 2,000,000 rows
    ... processed 2,500,000 rows
    ... processed 3,000,000 rows
    ... processed 3,500,000 rows
    ... processed 4,000,000 rows
    ... processed 4,500,000 rows
    ... processed 5,000,000 rows
Full data pattern frequencies (n=5,034,616 rows):
Pattern                      Count  Percentage  Sample vs Full
---------------------------------------------------------------------------
all_caps_word         :  1,001,487      19.89%  +0.00% diff
extra_whitespace      :    554,392      11.01%  +0.00% diff
url_in_name           :    176,144       3.50%  +0.00% diff
duplicate_adj_word    : 

## 8. Noise catalog — prioritised summary

Run all cells above, then populate the catalog below with observed frequencies.
This feeds directly into `01_normalize.ipynb` to decide which rules to build.

In [49]:
catalog_data = []

# 1. Collect noise pattern frequencies from Section 5
print("="*80)
print("AUTOMATED NOISE CATALOG")
print("="*80)
print("\nCollecting pattern frequencies from scans...\n")

for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if NAME_COL not in df.columns:
        continue
    col = df[NAME_COL].fillna("")
    
    for pat_name, pat in NOISE_PATTERNS.items():
        mask = col.str.contains(pat, na=False)
        n = mask.sum()
        pct = n / len(df) * 100
        
        # Determine priority
        if pct >= 10:
            priority = "HIGH"
        elif pct >= 1:
            priority = "MEDIUM"
        elif pct >= 0.1:
            priority = "LOW"
        else:
            priority = "TRACE"
        
        # Map pattern to normalization action
        action_map = {
            "hashtag_prefix": "strip prefix",
            "appended_digits": "strip suffix",
            "url_in_name": "remove URL fragments",
            "duplicate_adj_word": "collapse duplicates",
            "extra_whitespace": "normalize whitespace",
            "null_literal": "remove 'null' strings",
            "angle_brackets": "strip brackets",
            "all_caps_word": "case normalization",
            "dba_alias": "split into legal_name + alias_name",
            "parenthetical_sfx": "strip parentheses, normalize suffix",
            "accent_anomaly": "selective accent stripping",
            "pipe_separator": "remove or split on pipe",
            "leetspeak_digit": "flag for manual review",
        }
        
        if n > 0:  # Only include patterns that were found
            catalog_data.append({
                "pattern": pat_name,
                "source": src_name,
                "count": n,
                "pct": pct,
                "action": action_map.get(pat_name, "TBD"),
                "priority": priority,
                "sample_size": len(df)
            })


AUTOMATED NOISE CATALOG




In [50]:
# 2. Add address pattern frequencies
for src_name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    if ADDR_COL not in df.columns:
        continue
    col = df[ADDR_COL].fillna("")
    
    for pat_name, pat in ADDR_PATTERNS.items():
        if pat_name in ("all_caps", "reordered_addr", "empty"):
            mask = col.str.match(pat)
        else:
            mask = col.str.contains(pat, na=False)
        n = mask.sum()
        pct = n / len(df) * 100
        
        if pct >= 10:
            priority = "HIGH"
        elif pct >= 1:
            priority = "MEDIUM"
        elif pct >= 0.1:
            priority = "LOW"
        else:
            priority = "TRACE"
        
        action_map_addr = {
            "null_literal": "remove 'null' strings",
            "po_box": "extract PO Box, keep separate field",
            "all_caps": "case normalization",
            "reordered_addr": "parse and reorder components",
            "empty": "flag for name-only matching",
            "non_latin_chars": "preserve for embedding, transliterate for token matching",
        }
        
        if n > 0:
            catalog_data.append({
                "pattern": f"addr:{pat_name}",
                "source": src_name,
                "count": n,
                "pct": pct,
                "action": action_map_addr.get(pat_name, "TBD"),
                "priority": priority,
                "sample_size": len(df)
            })

In [51]:
# 3. Add cross-script stats if available
if 'cross_script' in dir() and 'total_pairs' in dir() and total_pairs > 0:
    cross_script_pct = (cross_script / total_pairs) * 100
    priority = "HIGH" if cross_script_pct >= 5 else "MEDIUM"
    catalog_data.append({
        "pattern": "cross_script_pairs",
        "source": "GT",
        "count": cross_script,
        "pct": cross_script_pct,
        "action": "embedding-based blocking + matching",
        "priority": priority,
        "sample_size": total_pairs
    })

In [53]:
# Build and display catalog
if catalog_data:
    catalog_df = pd.DataFrame(catalog_data)
    
    # Sort by priority then percentage
    priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2, "TRACE": 3}
    catalog_df['_priority_sort'] = catalog_df['priority'].map(priority_order)
    catalog_df = catalog_df.sort_values(['_priority_sort', 'pct'], ascending=[True, False])
    catalog_df = catalog_df.drop('_priority_sort', axis=1)
    
    # Display by priority level
    print("\n" + "="*80)
    print("HIGH PRIORITY PATTERNS (>10% or critical for matching)")
    print("="*80)
    high_priority = catalog_df[catalog_df['priority'] == 'HIGH']
    if len(high_priority) > 0:
        for _, row in high_priority.iterrows():
            print(f"{row['pattern']:<30} | {row['source']:<4} | {row['count']:>8,} ({row['pct']:>6.2f}%) | {row['action']}")
    else:
        print("  No high-priority patterns found")
    
    print("\n" + "="*80)
    print("MEDIUM PRIORITY PATTERNS (1-10% prevalence)")
    print("="*80)
    med_priority = catalog_df[catalog_df['priority'] == 'MEDIUM']
    if len(med_priority) > 0:
        for _, row in med_priority.iterrows():
            print(f"{row['pattern']:<30} | {row['source']:<4} | {row['count']:>8,} ({row['pct']:>6.2f}%) | {row['action']}")
    else:
        print("  No medium-priority patterns found")
    
    print("\n" + "="*80)
    print("LOW PRIORITY PATTERNS (0.1-1% prevalence)")
    print("="*80)
    low_priority = catalog_df[catalog_df['priority'] == 'LOW']
    if len(low_priority) > 0:
        for _, row in low_priority.iterrows():
            print(f"{row['pattern']:<30} | {row['source']:<4} | {row['count']:>8,} ({row['pct']:>6.2f}%) | {row['action']}")
    else:
        print("  No low-priority patterns found")
    
    # Summary statistics
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Total patterns detected: {len(catalog_df)}")
    print(f"  HIGH priority:   {len(high_priority)}")
    print(f"  MEDIUM priority: {len(med_priority)}")
    print(f"  LOW priority:    {len(low_priority)}")
    print(f"\n? Catalog complete. Use this to prioritize normalization rules in 01_normalize.ipynb")
    
    # Export catalog for reference
    catalog_export = catalog_df[['pattern', 'source', 'pct', 'action', 'priority']].copy()
    print(f"\nMarkdown table export:\n")
    # Try to use tabulate if available, otherwise use manual formatting
    try:
        print(catalog_export.to_markdown(index=False))
    except ImportError:
        # Manual markdown table formatting (tabulate not installed)
        print('| pattern | source | pct | action | priority |')
        print('|---------|--------|-----|--------|----------|')
        for _, row in catalog_export.iterrows():
            print(f'| {row["pattern"]} | {row["source"]} | {row["pct"]:.2f} | {row["action"]} | {row["priority"]} |')
else:
    print("No patterns collected. Run the scan cells above first.")


HIGH PRIORITY PATTERNS (>10% or critical for matching)
addr:all_caps                  | S2   | 2,598,431 ( 51.61%) | case normalization
all_caps_word                  | S2   | 1,001,487 ( 19.89%) | case normalization
extra_whitespace               | S2   |  554,392 ( 11.01%) | normalize whitespace
extra_whitespace               | S3   |  575,616 ( 10.89%) | normalize whitespace
cross_script_pairs             | GT   |   12,557 (  7.26%) | embedding-based blocking + matching

MEDIUM PRIORITY PATTERNS (1-10% prevalence)
addr:non_latin_chars           | S2   |  478,453 (  9.50%) | preserve for embedding, transliterate for token matching
addr:non_latin_chars           | S3   |  476,588 (  9.02%) | preserve for embedding, transliterate for token matching
accent_anomaly                 | S3   |  328,213 (  6.21%) | selective accent stripping
accent_anomaly                 | S2   |  290,263 (  5.77%) | selective accent stripping
addr:reordered_addr            | S1   |   84,634 (  3.84%) | par

## Next steps

1. Review counts above — mark HIGH / MEDIUM / LOW priority for normalization.
2. Hand off frequency table to `01_normalize.ipynb`.
3. Cross-script % -> informs embedding blocking budget in `02_blocking.ipynb`.
4. Address missing rate -> informs address fallback strategy.
5. Update `Documentation_template.md` Problem Analysis section with these numbers.
6. Note which patterns to instrument as toggleable rules (Step 1.1 rule-level instrumentation).